# Blockchain Lab 1 – Akcasu

This notebook implements core blockchain concepts step by step, starting from SHA-256 hashing through Proof of Work mining and chain forking.

In [ ]:
# ── Part 1: Imports and SHA-256 Hash Function ────────────────────────────────
import hashlib
import json
from copy import deepcopy

def sha256_text(text):
    """Return the SHA-256 hex digest of a UTF-8 encoded string."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

transactions = [
    "Alice pays Bob 10",
    "Bob pays Carol 4",
    "Carol pays David 2"
]

for transaction in transactions:
    print(transaction)
    print(sha256_text(transaction))
    print()

In [ ]:
# ── Part 1: Hash Change Test ─────────────────────────────────────────────────
# Demonstrates avalanche effect: a tiny input change produces a completely
# different hash digest.
original = "Alice pays Bob 10"
changed  = "Alice pays Bob 11"

print("Original Hash:", sha256_text(original))
print("Changed Hash: ", sha256_text(changed))

In [ ]:
# ── Part 2: Block Structure and Chain Linkage ────────────────────────────────
def calculate_block_hash(block):
    """Compute SHA-256 over all block fields except the stored hash."""
    data    = {key: value for key, value in block.items() if key != "hash"}
    encoded = json.dumps(data, sort_keys=True, separators=(",", ":"))
    return sha256_text(encoded)

def make_block(index, transactions, previous_hash, timestamp, nonce=0):
    """Create and return a new block dictionary with its hash computed."""
    block = {
        "index":         index,
        "timestamp":     timestamp,
        "transactions":  transactions,
        "previous_hash": previous_hash,
        "nonce":         nonce
    }
    block["hash"] = calculate_block_hash(block)
    return block

# Genesis block (block 0)
genesis = make_block(
    0,
    ["Course blockchain begins"],
    "0" * 64,
    "2026-10-01T09:00:00Z"
)

block_1 = make_block(
    1,
    ["Alice pays Bob 10", "Bob pays Carol 4"],
    genesis["hash"],
    "2026-10-01T09:01:00Z"
)

block_2 = make_block(
    2,
    ["Carol pays David 2"],
    block_1["hash"],
    "2026-10-01T09:02:00Z"
)

chain = [genesis, block_1, block_2]

def show_chain(blocks):
    """Print a summary row for each block in the chain."""
    for block in blocks:
        print(
            "Block", block["index"],
            "| Hash:",     block["hash"][:12],
            "| Previous:", block["previous_hash"][:12],
            "| Nonce:",    block["nonce"]
        )

show_chain(chain)

In [ ]:
# ── Part 3: Chain Validation and Tamper Detection ────────────────────────────
def validate_chain(blocks):
    """Return (True, message) if the chain is intact, (False, reason) otherwise."""
    for position, block in enumerate(blocks):
        recalculated = calculate_block_hash(block)
        if block["hash"] != recalculated:
            return False, f"Block {position} has changed contents"
        if position > 0:
            expected_previous = blocks[position - 1]["hash"]
            if block["previous_hash"] != expected_previous:
                return False, f"Block {position} has a broken previous-hash link"
    return True, "Chain is valid"

print("Original chain validation:  ", validate_chain(chain))

# Tamper with block 1 transaction data
tampered_chain = deepcopy(chain)
tampered_chain[1]["transactions"][0] = "Alice pays Bob 100"
print("Tampered chain validation:  ", validate_chain(tampered_chain))

# Re-hash only the tampered block (attacker patches just one hash)
tampered_chain[1]["hash"] = calculate_block_hash(tampered_chain[1])
print("After re-hashing block 1:   ", validate_chain(tampered_chain))

In [ ]:
# ── Part 4: Proof of Work Mining ─────────────────────────────────────────────
def mine_block(index, transactions, previous_hash, timestamp, difficulty=3):
    """
    Increment nonce until the block hash starts with `difficulty` leading zeros.
    Returns the mined block dict and the number of attempts taken.
    """
    block = {
        "index":         index,
        "timestamp":     timestamp,
        "transactions":  transactions,
        "previous_hash": previous_hash,
        "nonce":         0
    }
    target   = "0" * difficulty
    attempts = 0

    while True:
        attempts += 1
        digest = calculate_block_hash(block)
        if digest.startswith(target):
            block["hash"] = digest
            return block, attempts
        block["nonce"] += 1

mined_block, attempts = mine_block(
    3,
    ["David pays Erin 1"],
    chain[-1]["hash"],
    "2026-10-01T09:03:00Z",
    difficulty=3
)

print("Attempts:", attempts)
print("Nonce:   ", mined_block["nonce"])
print("Hash:    ", mined_block["hash"])

In [ ]:
# ── Part 5: Forking and Longest-Chain Selection ───────────────────────────────
branch_a = deepcopy(chain)
branch_b = deepcopy(chain)

# Two competing blocks mined on top of the same parent
block_a, attempts_a = mine_block(
    3, ["David pays Erin 1"], branch_a[-1]["hash"],
    "2026-10-01T09:03:00Z", difficulty=3
)

block_b, attempts_b = mine_block(
    3, ["David pays Frank 1"], branch_b[-1]["hash"],
    "2026-10-01T09:03:01Z", difficulty=3
)

branch_a.append(block_a)
branch_b.append(block_b)

print("Branch A validity:", validate_chain(branch_a))
print("Branch B validity:", validate_chain(branch_b))

# Extend branch A by one more block — it becomes the longest chain
block_a_2, attempts_a_2 = mine_block(
    4, ["Erin pays Grace 1"], branch_a[-1]["hash"],
    "2026-10-01T09:04:00Z", difficulty=3
)
branch_a.append(block_a_2)

preferred_branch = max([branch_a, branch_b], key=len)
print("Preferred chain length:", len(preferred_branch))
show_chain(preferred_branch)

In [ ]:
# ── Part 5 – Extension: Transaction Validation Rules ─────────────────────────
def is_valid_transaction(tx):
    """
    Validate logical rules for a transaction string of the form
    '<Sender> pays <Receiver> <Amount>'.
    Returns True only when the amount is positive and sender != receiver.
    """
    try:
        parts = tx.split(" ")
        if len(parts) != 4 or parts[1] != "pays":
            return False
        sender, _, receiver, amount_str = parts
        amount = float(amount_str)
        if amount <= 0 or sender == receiver:
            return False
        return True
    except Exception:
        return False

# A block can be cryptographically valid yet contain a logically invalid transaction
invalid_tx = "Alice pays Bob -10"
bad_block   = make_block(
    3,
    [invalid_tx],
    chain[-1]["hash"],
    "2026-10-01T09:05:00Z"
)

test_chain = deepcopy(chain)
test_chain.append(bad_block)

print("Cryptographic chain validation:", validate_chain(test_chain))
print("Logical transaction validation:", is_valid_transaction(bad_block["transactions"][0]))